# Classification & Scoring

Classification

In [2]:
import pandas as pd
import numpy as np
import hvplot.pandas
import matplotlib
import statsmodels.api as sm
import math
from statsmodels.regression.rolling import RollingOLS
import holoviews as hv
import plotly.express as px
import plotly.figure_factory as ff

#benchmarks

benchmark = pd.read_csv('fixed_income_benchmark_data_2024.csv', index_col=0, encoding='latin-1', header=None)
benchmark.columns = benchmark.iloc[0]
benchmark = benchmark.iloc[2:]
benchmark.index = pd.to_datetime(benchmark.index)
benchmark = benchmark.apply(lambda x: pd.to_numeric(x, errors='coerce'))

#data
data = pd.read_csv('fixed_income_universe_raw_data_12092024.csv', index_col=0, encoding='latin-1', header=None)
non_trilogy_mask = data.loc['Fund'].str.contains('Trilogy')
data_without_triology = data.loc[:,~non_trilogy_mask]
transposed_and_with_daily_pricing = data_without_triology.T[data_without_triology.T['Pricing Frequency'] == 'Daily']
transposed_and_with_daily_pricing['APIR Code'] = transposed_and_with_daily_pricing['APIR Code'].fillna(transposed_and_with_daily_pricing['Ticker'])
data_for_std_calc = transposed_and_with_daily_pricing.T

#for cash too:
categorical_data = transposed_and_with_daily_pricing.iloc[:,:8]
categorical_data['is_it_cash'] = categorical_data['Fund'].apply(lambda x: 'yes' if 'Cash' in x else 'no')
categorical_data = categorical_data.set_index('APIR Code')

#for stdev:
data_with_tall_columns = data_for_std_calc.copy()
data_with_tall_columns.columns = [data_for_std_calc.iloc[0], data_for_std_calc.iloc[1]]
data_with_tall_columns = data_with_tall_columns.iloc[8:]
data_with_tall_columns = data_with_tall_columns.apply(lambda x: pd.to_numeric(x, errors='coerce'))
data_with_tall_columns = data_with_tall_columns/100
df_tall_columns_for_std_dev = pd.DataFrame(data_with_tall_columns.std()*math.sqrt(12))
df_tall_columns_for_std_dev = df_tall_columns_for_std_dev.droplevel(axis=0,level=0)*100
df_tall_columns_for_std_dev = df_tall_columns_for_std_dev.rename({0:'annualized_vol'},axis=1)
df_tall_columns_for_std_dev.sort_values(by='annualized_vol').dropna()[0:20]


global_dataset = transposed_and_with_daily_pricing[transposed_and_with_daily_pricing['Global Category'].str.contains("Global|US|Emerging")].T
aus_dataset = transposed_and_with_daily_pricing[transposed_and_with_daily_pricing['Global Category'].str.contains("Australia")].T
misc_funds = transposed_and_with_daily_pricing[transposed_and_with_daily_pricing['Global Category'] =='Fixed Income Miscellaneous']['Fund']

global_dataset.columns = global_dataset.iloc[1]
aus_dataset.columns = aus_dataset.iloc[1]

global_dataset = global_dataset.iloc[8:]
global_dataset_partially_cleaned = global_dataset.apply(lambda x: pd.to_numeric(x, errors='coerce'))

aus_dataset = aus_dataset.iloc[8:]
aus_dataset_cleaned = aus_dataset.apply(lambda x: pd.to_numeric(x, errors='coerce'))

aus_dataset_cleaned = aus_dataset_cleaned/100
global_dataset_cleaned1 = global_dataset_partially_cleaned/100

aus_dataset_cleaned.index = pd.to_datetime(aus_dataset_cleaned.index)
global_dataset_cleaned1.index = pd.to_datetime(global_dataset_cleaned1.index)


#BENCHMARKS
benchmark_returns = benchmark.pct_change()
new_purified_benchmarks = pd.DataFrame(index = benchmark_returns.index)
new_purified_benchmarks['Risk Free Return'] = benchmark_returns['Ausbond Bank Bill']
new_purified_benchmarks['Domestic Duration'] = benchmark_returns['Ausbond Government'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Ausbond Composite minus RF'] = benchmark_returns['Ausbond Composite'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Ausbond Government minus RF'] = benchmark_returns['Ausbond Government'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Ausbond FRN minus RF'] = benchmark_returns['Ausbond Credit FRN'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Ausbond Credit minus RF'] = benchmark_returns['Ausbond Credit'] - benchmark_returns['Ausbond Bank Bill']

benchmark_returns['Global Agg minus RF'] = benchmark_returns['Global Agg Unhedged'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Global Treasury minus RF'] = benchmark_returns['Global Agg Treasury Unhedged'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Global Credit minus RF'] = benchmark_returns['Global Agg Credit Unhedged'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['Global HY minus RF'] = benchmark_returns['Global High Yield Unhedged'] - benchmark_returns['Ausbond Bank Bill']
benchmark_returns['EM debt minus RF'] = benchmark_returns['JPM EMBI '] - benchmark_returns['Ausbond Bank Bill']

#domestic credit with beta removal
exog = sm.add_constant(benchmark_returns['Ausbond Government'], prepend=False)
model = RollingOLS(benchmark_returns['Ausbond Credit'], exog, window = 24, missing='drop').fit()
rolling_beta_credit_domestic = model.params['Ausbond Government']
new_purified_benchmarks['Domestic Credit'] = benchmark_returns['Ausbond Credit'] - (benchmark_returns['Ausbond Government'] * rolling_beta_credit_domestic)

new_purified_benchmarks['Global Duration'] = benchmark_returns['Global Agg Treasury Unhedged'] - benchmark_returns['Ausbond Bank Bill']

#global credit with beta removal
exog1 = sm.add_constant(benchmark_returns['Global Agg Treasury Unhedged'], prepend=False)
model1 = RollingOLS(benchmark_returns['Global Agg Credit Unhedged'], exog1, window = 24, missing='drop').fit()
rolling_beta_credit_global = model1.params['Global Agg Treasury Unhedged']
new_purified_benchmarks['Global Credit'] = benchmark_returns['Global Agg Credit Unhedged'] - (benchmark_returns['Global Agg Treasury Unhedged'] * rolling_beta_credit_global)

#high yield beta removal - duration
exog2 = sm.add_constant(benchmark_returns['Global Agg Treasury Unhedged'], prepend=False)
model2 = RollingOLS(benchmark_returns['Global High Yield Unhedged'], exog2, window = 24, missing='drop').fit()
rolling_beta_hy_global = model2.params['Global Agg Treasury Unhedged']

# credit removal
exog3 = sm.add_constant(benchmark_returns['Global Agg Credit Unhedged'], prepend=False)
model3 = RollingOLS(benchmark_returns['Global High Yield Unhedged'], exog3, window = 24, missing='drop').fit()
rolling_beta_hy_global_for_credit = model3.params['Global Agg Credit Unhedged']

new_purified_benchmarks['Global High Yield'] = benchmark_returns['Global High Yield Unhedged'] - (benchmark_returns['Global Agg Treasury Unhedged'] * rolling_beta_hy_global) #- (benchmark_returns['Global Agg Credit Unhedged'] * rolling_beta_hy_global_for_credit)

#FRN
new_purified_benchmarks['Domestic FRN'] = benchmark_returns['Ausbond Credit FRN'] - benchmark_returns['Ausbond Bank Bill']

#Currency
new_purified_benchmarks['Currency'] = benchmark_returns['S&P 500 AUD Hedged'] - benchmark_returns['S&P 500']

#EM Debt
new_purified_benchmarks['EM Debt'] = benchmark_returns['JPM EMBI '] - benchmark_returns['Ausbond Bank Bill']

global_benchmarks = new_purified_benchmarks[['Global Duration','Global Credit','Global High Yield','Currency','EM Debt']]
aussie_benchmarks = new_purified_benchmarks[['Domestic Credit','Domestic Duration','Domestic FRN','Currency']]

C:\Users\JohnBilsel\AppData\Local\Temp\ipykernel_32164\309337777.py:17: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  benchmark.index = pd.to_datetime(benchmark.index)
C:\Users\JohnBilsel\AppData\Local\Temp\ipykernel_32164\309337777.py:61: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  aus_dataset_cleaned.index = pd.to_datetime(aus_dataset_cleaned.index)
C:\Users\JohnBilsel\AppData\Local\Temp\ipykernel_32164\309337777.py:62: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  global_dataset_cleaned1.index = pd.to_datetime(global_dataset_cleaned1.index)


Aussie:

In [3]:
excess_return = aus_dataset_cleaned.sub((aussie_benchmarks[['Domestic Credit','Domestic Duration']]).mean(axis=1), axis=0)
tracking_errors = excess_return.std() * math.sqrt(12)

aus_dataset_cleaned_new_dates = aus_dataset_cleaned.loc[aussie_benchmarks.index.min():]

#adjust manager returns w/ risk-free rate
aussie_manager_returns_minus_rf = aus_dataset_cleaned_new_dates.sub(new_purified_benchmarks['Risk Free Return'], axis=0)

# beta estimation
output = {}
output2 = {}
for i in aussie_manager_returns_minus_rf.columns:
    y = aussie_manager_returns_minus_rf[i]
    x = aussie_benchmarks
    if y.count() < 24:
        output[i] = np.nan
        output2[i] = np.nan
    else: 
        model = sm.OLS(y, sm.add_constant(x), missing='drop').fit()
        output[i] = model.params
        output2[i] = model.tvalues

coefficients = pd.DataFrame(output)
tstats = pd.DataFrame(output2)
coefficients.columns = aussie_manager_returns_minus_rf.columns
tstats.columns = aussie_manager_returns_minus_rf.columns

classifications = pd.DataFrame(index=tstats.columns, columns=['classification_1', 'classification_2'])

for i in tstats.columns:
    ranks = tstats[i].sort_values(ascending=False).drop('const', axis=0)  # sort t-stats and drop constant term

    # Combined conditions for 'Cash' classification
    if (categorical_data.loc[i, 'is_it_cash'] == 'yes' or 
        df_tall_columns_for_std_dev.loc[i, 'annualized_vol'] < 0.5 or 
        ranks.name in ['WFS377AU', 'PDL8847AU']):
        classifications.at[i, 'classification_1'] = 'Cash'
        classifications.at[i, 'classification_2'] = 'nan'

    # Currency-related classification
    elif ranks.index[0] == 'Currency':
        if ranks.iloc[1] > 2.581:
            classifications.at[i, 'classification_1'] = ranks.index[1] if ranks.iloc[1] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else 'Alpha' if tracking_errors.loc[i] > 0.02 else 'nan' 
            classifications.at[i, 'classification_2'] = ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[2] != 'EM Debt' and ranks.index[2] != 'Currency' else ranks.index[3] if ranks.iloc[3] > 2.581 and ranks.index[3] != 'EM Debt' and ranks.index[3] != 'Currency' else 'nan' 

        elif tracking_errors.loc[i] < 0.02:
            classifications.at[i, 'classification_1'] = 'Beta'
            classifications.at[i, 'classification_2'] = 'nan'

            # Default to 'Alpha'
        else:
            classifications.at[i, 'classification_1'] = 'Alpha'
            classifications.at[i, 'classification_2'] = 'nan'

    # General t-stat classification
    elif ranks.iloc[0] > 2.581:
        classifications.at[i, 'classification_1'] = ranks.index[0]
        classifications.at[i, 'classification_2'] = ranks.index[1] if ranks.iloc[1] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else ranks.index[2] if ranks.iloc[2] > 2.581 else 'nan' 

    # Beta classification
    elif tracking_errors.loc[i] < 0.02:
        classifications.at[i, 'classification_1'] = 'Beta'
        classifications.at[i, 'classification_2'] = 'nan'

    # Default to 'Alpha'
    else:
        classifications.at[i, 'classification_1'] = 'Alpha'
        classifications.at[i, 'classification_2'] = 'nan'
    
final_df = tstats.T
final_df['Classification'] = classifications['classification_1']
final_df['Classification #2'] = classifications['classification_2']
final_df['Tracking Error'] = tracking_errors

#number of months w/ data
masker = excess_return.copy()
masker.columns = aussie_manager_returns_minus_rf.columns
final_df['Number of Months with Data'] = masker.count(axis=0)

final_df_labelled_aussie = final_df.reset_index().merge(transposed_and_with_daily_pricing[['APIR Code', 'Fund']], on='APIR Code', how='left')
final_df_labelled_aussie.index = final_df_labelled_aussie['Fund']

Global:

In [4]:
global_dataset_cleaned = global_dataset_cleaned1.drop(['ETL521AU','DFA1AU'],axis=1)
excess_return1 = global_dataset_cleaned.sub((global_benchmarks[['Global Credit','Global Duration']]).mean(axis=1), axis=0)
tracking_errors = excess_return1.std() * math.sqrt(12)

global_dataset_cleaned_new_dates = global_dataset_cleaned.loc[global_benchmarks.index.min():]

#adjust manager returns w/ risk-free rate
global_manager_returns_minus_rf = global_dataset_cleaned_new_dates.sub(new_purified_benchmarks['Risk Free Return'], axis=0)

# beta estimation
output3 = {}
output4 = {}
for i in global_manager_returns_minus_rf.columns:
    y = global_manager_returns_minus_rf[i]
    x = global_benchmarks
    if y.count() < 24:
        output3[i] = np.nan
        output4[i] = np.nan
    else: 
        model = sm.OLS(y, sm.add_constant(x), missing='drop').fit()
        output3[i] = model.params
        output4[i] = model.tvalues

coefficients = pd.DataFrame(output3)
tstats = pd.DataFrame(output4)
coefficients.columns = global_manager_returns_minus_rf.columns
tstats.columns = global_manager_returns_minus_rf.columns

#classification
classifications = pd.DataFrame(index=tstats.columns, columns=['classification_1', 'classification_2'])

for i in tstats.columns:
    ranks = tstats[i].sort_values(ascending=False).drop('const', axis=0)  # sort t-stats and drop constant term

    # Combined conditions for 'Cash' classification
    if (categorical_data.loc[i, 'is_it_cash'] == 'yes' or 
        df_tall_columns_for_std_dev.loc[i, 'annualized_vol'] < 0.5 or 
        ranks.name in ['WFS377AU', 'PDL8847AU']):
        classifications.at[i, 'classification_1'] = 'Cash'
        classifications.at[i, 'classification_2'] = 'nan'

    # Currency-related classification
    elif ranks.index[0] == 'Currency':
        if ranks.iloc[1] > 2.581:
            classifications.at[i, 'classification_1'] = ranks.index[1] if ranks.iloc[1] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else 'Alpha' if tracking_errors.loc[i] > 0.02 else 'nan' 
            classifications.at[i, 'classification_2'] = ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[2] != 'EM Debt' and ranks.index[2] != 'Currency' else ranks.index[3] if ranks.iloc[3] > 2.581 and ranks.index[3] != 'EM Debt' and ranks.index[3] != 'Currency' else 'nan' 

        elif tracking_errors.loc[i] < 0.02:
            classifications.at[i, 'classification_1'] = 'Beta'
            classifications.at[i, 'classification_2'] = 'nan'

            # Default to 'Alpha'
        else:
            classifications.at[i, 'classification_1'] = 'Alpha'
            classifications.at[i, 'classification_2'] = 'nan'
    
    # EM-related classification
    elif ranks.index[0] == 'EM Debt':
        if ranks.iloc[1] > 2.581:
            classifications.at[i, 'classification_1'] = ranks.index[1] if ranks.iloc[1] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else 'Alpha' if tracking_errors.loc[i] > 0.02 else 'nan' 
            classifications.at[i, 'classification_2'] = ranks.index[2] if ranks.iloc[2] > 2.581 and ranks.index[2] != 'EM Debt' and ranks.index[2] != 'Currency' else ranks.index[3] if ranks.iloc[3] > 2.581 and ranks.index[3] != 'EM Debt' and ranks.index[3] != 'Currency' else 'nan' 

        elif tracking_errors.loc[i] < 0.02:
            classifications.at[i, 'classification_1'] = 'Beta'
            classifications.at[i, 'classification_2'] = 'nan'

            # Default to 'Alpha'
        else:
            classifications.at[i, 'classification_1'] = 'Alpha'
            classifications.at[i, 'classification_2'] = 'nan'

    # General t-stat classification
    elif ranks.iloc[0] > 2.581:
        classifications.at[i, 'classification_1'] = ranks.index[0]
        classifications.at[i, 'classification_2'] = ranks.index[1] if ranks.iloc[1] > 2.581 and ranks.index[1] != 'EM Debt' and ranks.index[1] != 'Currency' else ranks.index[2] if ranks.iloc[2] > 2.581 else 'nan' 

    # Beta classification
    elif tracking_errors.loc[i] < 0.02:
        classifications.at[i, 'classification_1'] = 'Beta'
        classifications.at[i, 'classification_2'] = 'nan'

    # Default to 'Alpha'
    else:
        classifications.at[i, 'classification_1'] = 'Alpha'
        classifications.at[i, 'classification_2'] = 'nan'
    
final_df = tstats.T
final_df['Classification'] = classifications['classification_1']
final_df['Classification #2'] = classifications['classification_2']
final_df['Tracking Error'] = tracking_errors

#number of months w/ data
masker = excess_return1.copy()
masker.columns = global_manager_returns_minus_rf.columns
final_df['Number of Months with Data'] = masker.count(axis=0)

final_df_labelled_global = final_df.reset_index().merge(transposed_and_with_daily_pricing[['APIR Code', 'Fund']], on='APIR Code', how='left')
final_df_labelled_global.index = final_df_labelled_global['Fund']

In [50]:
search = 'janus'
#final_df_labelled_global[final_df_labelled_global.index.to_frame().apply(lambda x: x.astype(str).str.contains(search, case=False)).any(axis=1)]
final_df_labelled_aussie[final_df_labelled_aussie.index.to_frame().apply(lambda x: x.astype(str).str.contains(search, case=False)).any(axis=1)]

,APIR Code,const,Domestic Credit,Domestic Duration,Domestic FRN,Currency,Classification,Classification #2,Tracking Error,Number of Months with Data,Fund
Fund,,,,,,,,,,,
Janus Henderson Tactical Income Actv ETF,HGI7649AU,0.286496,1.426548,5.439303,0.671066,3.479432,Domestic Duration,nan,0.023996,49,Janus Henderson Tactical Income Actv ETF
Janus Henderson Diversified Credit,IOF127AU,-2.776924,3.338597,2.215726,8.118085,4.448603,Domestic FRN,Domestic Credit,0.029952,142,Janus Henderson Diversified Credit
Janus Henderson Cnsrv Fxd Intst,IOF47AU,-3.021813,5.298798,-0.070280,15.216959,1.843229,Domestic FRN,Domestic Credit,0.022005,244,Janus Henderson Cnsrv Fxd Intst
Janus Henderson Cnsrv Fxd Intst Instl,IOF111AU,-3.987090,7.652643,0.124323,18.669362,3.259640,Domestic FRN,Domestic Credit,0.022098,244,Janus Henderson Cnsrv Fxd Intst Instl
Janus Henderson Australian Fxd Intst,IOF46AU,-4.601920,6.651501,59.282515,5.100682,3.155117,Domestic Duration,Domestic Credit,0.018847,244,Janus Henderson Australian Fxd Intst
Janus Henderson Tactical Income Instl,HGI4188AU,NaN,NaN,NaN,NaN,NaN,Alpha,nan,NaN,1,Janus Henderson Tactical Income Instl
Janus Henderson AUS Fxd Intst MPS,WFS1859AU,-1.069895,5.449279,62.801035,-0.390356,2.798813,Domestic Duration,Domestic Credit,0.030038,70,Janus Henderson AUS Fxd Intst MPS
Janus Henderson Australian Fxd IntstInst,IOF113AU,-4.110683,6.739114,59.052514,4.685771,2.967061,Domestic Duration,Domestic Credit,0.018784,244,Janus Henderson Australian Fxd IntstInst
Janus Henderson Tactical Income,IOF145AU,-1.535477,2.872648,10.108117,6.932582,4.276624,Domestic Duration,Domestic FRN,0.018640,182,Janus Henderson Tactical Income


Rolling visualization:

In [70]:
fund_wanted = 'VGB'

use_for_vis = aus_dataset_cleaned_new_dates #.droplevel(axis=1,level=0) # aus_dataset_cleaned_new_dates

exog = sm.add_constant(aussie_benchmarks, prepend=False)
mod = RollingOLS(use_for_vis[fund_wanted], exog, window=36, missing='drop').fit()
plotter = mod.params.dropna().reset_index().drop('const',axis=1)
fig = px.line(plotter.iloc[3:], x=0, y=plotter.iloc[:,1:].columns, title=f'Janus')
fig.update_xaxes(title_text='Date')
fig.update_yaxes(title_text='Returns')
fig.show()

c:\Users\JohnBilsel\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\regression\rolling.py:255: RuntimeWarning:

divide by zero encountered in scalar divide



Actual Scoring:

In [ ]:
classification = final_df_labelled_aussie[['APIR Code','Classification','Classification #2']]
classification.index = classification['APIR Code']

#beta estimation
output = {}
output2 = {}

data_drop_level_0 = aussie_manager_returns_minus_rf

for i in data_drop_level_0.columns:
    y = data_drop_level_0.loc[:,i].iloc[-120:]
    
    if classification.loc[i]['Classification'] == 'Beta' or classification.loc[i]['Classification'] == 'Alpha':
        x = benchmark_returns.loc[:,'Ausbond Bank Bill'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Domestic Duration':
        x = benchmark_returns.loc[:,'Ausbond Government minus RF'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Domestic Credit':
        x = benchmark_returns.loc[:,'Ausbond Credit minus RF'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Cash':
        x = benchmark_returns.loc[:,'Ausbond Bank Bill'].iloc[-120:]

    if y.count() < 24: 
            output[i] = np.nan
            output2[i] = np.nan
        
    else: 
        model = sm.OLS(y, sm.add_constant(x), missing='drop').fit()
        output[i] = model.params
        output2[i] = model.rsquared

coefficients = pd.DataFrame(output)
coefficients.columns = aus_dataset_cleaned_new_dates.columns
coefficients = coefficients.T
coefs = coefficients.reset_index()
coefs.index = aus_dataset_cleaned_new_dates.columns
coefs['classification'] = classification['Classification']
coefs = coefs.fillna(0)

#seperate benchmark df and fund returns
benchmark_df = aussie_benchmarks
fund_returns_df = data_drop_level_0.iloc[-120:]

#align column names
coefs1 = coefs

#create the monthly alpha with a static beta
bx = pd.DataFrame(index = benchmark_df.iloc[-120:].index, columns = coefs1.iloc[-120:].index)
for j in coefs1.index:
    fund_betas = coefs1.loc[j]
    bx[j] = (benchmark_df.iloc[-120:] * fund_betas).sum(axis=1)

bx1 = bx[fund_returns_df.columns]
final_alpha_df = fund_returns_df - bx1
final_alpha_df.columns = aus_dataset_cleaned_new_dates.columns

#std of alpha
std_of_alpha = final_alpha_df.std()*math.sqrt(12)
std_of_alpha1 = pd.DataFrame(std_of_alpha)
std_of_alpha2 = std_of_alpha1.reset_index().set_index('APIR Code')

#track record and positive months alpha
number_of_months_inception = final_alpha_df.count()
numb_of_pos_months = pd.DataFrame((final_alpha_df>0).sum())
numb_of_pos_months = numb_of_pos_months.reset_index().set_index('APIR Code')

number_of_months_inception = final_alpha_df.count()
number_of_months_inception1 = pd.DataFrame(number_of_months_inception)
number_of_months_inception1 = number_of_months_inception1.reset_index().set_index('APIR Code')

classification['Annual-Adjusted-CAPM-Alpha'] = coefs['const'] * 12
classification['Monthly-Adjusted-CAPM-Alpha'] = coefs['const']
classification['Annualized Std of Alpha'] = std_of_alpha2[0]
classification['No. of Positive Months Alpha'] = numb_of_pos_months[0]
classification['No. of Months with Data'] = number_of_months_inception1[0]
classification['Ratio of Positive Months Alpha to Total Months'] = classification['No. of Positive Months Alpha'] / classification['No. of Months with Data']

rounded_classif_aussie = round(classification[['Annual-Adjusted-CAPM-Alpha','Monthly-Adjusted-CAPM-Alpha', 'Annualized Std of Alpha',
                                        'No. of Positive Months Alpha','Ratio of Positive Months Alpha to Total Months']],5)
#add months
rounded_classif_aussie['No of Months with Data'] = aus_dataset_cleaned_new_dates.count(axis=0)

In [7]:
classification = final_df_labelled_global[['APIR Code','Classification','Classification #2']]
classification.index = classification['APIR Code']

#beta estimation
output = {}
output2 = {}

data_drop_level_0 = global_manager_returns_minus_rf

for i in data_drop_level_0.columns:
    y = data_drop_level_0.loc[:,i].iloc[-120:]
    
    if classification.loc[i]['Classification'] == 'Beta' or classification.loc[i]['Classification'] == 'Alpha':
        x = benchmark_returns.loc[:,'Ausbond Bank Bill'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Global Duration':
        x = benchmark_returns.loc[:,'Global Treasury minus RF'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Global Credit':
        x = benchmark_returns.loc[:,'Global Credit minus RF'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'Global High Yield':
        x = benchmark_returns.loc[:,'Global HY minus RF'].iloc[-120:]

    elif classification.loc[i]['Classification'] == 'EM Debt':
        x = benchmark_returns.loc[:,'EM debt minus RF'].iloc[-120:]

    if y.count() < 24: 
            output[i] = np.nan
            output2[i] = np.nan
        
    else: 
        model = sm.OLS(y, sm.add_constant(x), missing='drop').fit()
        output[i] = model.params
        output2[i] = model.rsquared

coefficients = pd.DataFrame(output)
coefficients.columns = global_dataset_cleaned_new_dates.columns
coefficients = coefficients.T
coefs = coefficients.reset_index()
coefs.index = global_dataset_cleaned_new_dates.columns
coefs['classification'] = classification['Classification']
coefs = coefs.fillna(0)

#seperate benchmark df and fund returns
benchmark_df = global_benchmarks
fund_returns_df = data_drop_level_0.iloc[-120:]

#align column names
coefs1 = coefs

#create the monthly alpha with a static beta
bx = pd.DataFrame(index = benchmark_df.iloc[-120:].index, columns = coefs1.iloc[-120:].index)
for j in coefs1.index:
    fund_betas = coefs1.loc[j]
    bx[j] = (benchmark_df.iloc[-120:] * fund_betas).sum(axis=1)

bx1 = bx[fund_returns_df.columns]
final_alpha_df = fund_returns_df - bx1
final_alpha_df.columns = global_dataset_cleaned_new_dates.columns

#std of alpha
std_of_alpha = final_alpha_df.std()*math.sqrt(12)
std_of_alpha1 = pd.DataFrame(std_of_alpha)
std_of_alpha2 = std_of_alpha1.reset_index().set_index('APIR Code')

#track record and positive months alpha
number_of_months_inception = final_alpha_df.count()
numb_of_pos_months = pd.DataFrame((final_alpha_df>0).sum())
numb_of_pos_months = numb_of_pos_months.reset_index().set_index('APIR Code')

number_of_months_inception = final_alpha_df.count()
number_of_months_inception1 = pd.DataFrame(number_of_months_inception)
number_of_months_inception1 = number_of_months_inception1.reset_index().set_index('APIR Code')

classification['Annual-Adjusted-CAPM-Alpha'] = coefs['const'] * 12
classification['Monthly-Adjusted-CAPM-Alpha'] = coefs['const']
classification['Annualized Std of Alpha'] = std_of_alpha2[0]
classification['No. of Positive Months Alpha'] = numb_of_pos_months[0]
classification['No. of Months with Data'] = number_of_months_inception1[0]
classification['Ratio of Positive Months Alpha to Total Months'] = classification['No. of Positive Months Alpha'] / classification['No. of Months with Data']

rounded_classif_global = round(classification[['Annual-Adjusted-CAPM-Alpha','Monthly-Adjusted-CAPM-Alpha', 'Annualized Std of Alpha',
                                        'No. of Positive Months Alpha','Ratio of Positive Months Alpha to Total Months']],5)
#add months
rounded_classif_global['No of Months with Data'] = global_dataset_cleaned_new_dates.count(axis=0)

In [8]:
def z_score_per_category(export_df):
  export_df = export_df[export_df['No of Months with Data']>24]
  weights = np.array([1/3,1/3, 1/6,1/6])
  dict_of_dfs = {}
  for i  in export_df['Classification'].unique():
      isolate = export_df[(export_df['Classification']==i) | (export_df['Classification #2']==i)]
      new_df = isolate[['Monthly-Adjusted-CAPM-Alpha', 
                                  'Annualized Std of Alpha', 
                                    'No. of Positive Months Alpha', 
                                      'Ratio of Positive Months Alpha to Total Months']]

      new_df['Annualized Std of Alpha'] = -new_df['Annualized Std of Alpha']
      new_df = (new_df-new_df.mean())/new_df.std()
      new_df['Overall Z-Score'] = new_df[['Monthly-Adjusted-CAPM-Alpha', 
                                      'Annualized Std of Alpha', 
                                      'No. of Positive Months Alpha', 
                                          'Ratio of Positive Months Alpha to Total Months']].dot(weights)

      min_val = new_df['Overall Z-Score'].min()
      max_val = new_df['Overall Z-Score'].max()
      isolate['Overall Score'] = (new_df['Overall Z-Score'] - min_val) / (max_val - min_val)
      isolate = isolate.reset_index()
      isolate_adjusted = (isolate[['Fund','Annual-Adjusted-CAPM-Alpha', 
                                'Annualized Std of Alpha', 
                                'No. of Positive Months Alpha', 
                                    'Ratio of Positive Months Alpha to Total Months',
                                    'Overall Score','Tracking Error','No of Months with Data']]).sort_values(by='Overall Score', ascending=False)
      dict_of_dfs[i] = isolate_adjusted
    
  return dict_of_dfs

aussie_scoring_export = rounded_classif_aussie.merge(final_df_labelled_aussie[['APIR Code','Fund','Classification','Classification #2','Tracking Error']], on='APIR Code',how='left').set_index('APIR Code')
global_scoring_export = rounded_classif_global.merge(final_df_labelled_global[['APIR Code','Fund','Classification','Classification #2','Tracking Error']], on='APIR Code',how='left').set_index('APIR Code')
aussie_export_final = z_score_per_category(aussie_scoring_export)
global_export_final = z_score_per_category(global_scoring_export)

In [12]:
from datetime import datetime
def export_dfs_to_excel(dict_of_dfs, output_file):
    with pd.ExcelWriter(output_file) as writer:
        for key, df in dict_of_dfs.items():
            # Write each DataFrame to a separate sheet named after the key
            df.to_excel(writer, sheet_name=key, index=False)

# Example usage
export_dfs_to_excel(global_export_final, f'global_scoring_final_{datetime.today().date()}.xlsx')
export_dfs_to_excel(aussie_export_final, f'aussie_scoring_final_{datetime.today().date()}.xlsx')